# Ch.1 — Linear Algebra: Lines, Weights, and Biases

**Running theme.** A basketball free throw. In the first 0.2 s after release, the ball rises almost in a straight line — the ideal regime to see what a weight $w$ and a bias $b$ actually *do*.

**Learning outcomes.** After running this notebook you will be able to:
1. Recognise the equation $y = w\,x + b$ in algebra, physics, and ML notation.
2. Drag $w$ and $b$ interactively and predict the effect without looking.
3. Compute a dot product three different ways in NumPy and see why all three give the same number.
4. Fit a line to noisy $(t, h)$ samples by eye and feel why we'll want an optimiser in Ch.3/Ch.4.

## 1 · Imports and setup

Standard stack: `numpy`, `matplotlib`, `ipywidgets`. The `%matplotlib widget` magic gives us live updates when sliders move; if your Jupyter install doesn't have `ipympl`, fall back to `%matplotlib inline` and re-run the cell manually.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, FloatText, HBox, VBox, Output, jslink

try:
    get_ipython().run_line_magic("matplotlib", "widget")  # interactive canvas
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")   # fallback

plt.rcParams.update({"figure.figsize": (8.5, 5.0), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})
np.random.seed(17)

## 2 · The equation of a line — three framings, one object

The same line, written three ways:

| Framing | Equation |
|---|---|
| Algebra | $y = m\,x + c$ |
| Physics | $h(t) = v\,t + h_0$ |
| ML | $\hat{y} = w\,x + b$ |

Run the cell below to confirm all three are interchangeable.

In [ ]:
x = np.linspace(-4, 4, 200)
w, b = 1.5, -0.5
y = w * x + b
print(f"line y = {w}*x + {b}")
print(f"  y(x=0) = {w*0 + b}     (so b = y-intercept)")
print(f"  slope  = (y(1)-y(0)) / (1-0) = {(w*1+b) - (w*0+b):.3f}   (so w = slope)")

## 3 · Interactive: drag $w$ and $b$

Two sliders + two text boxes, wired *bidirectionally*: move the slider and the number updates; type a number and the slider moves. The line in the plot re-renders live.

**Try it.** Before you touch anything, predict:

- What happens when $w$ goes from $+1$ to $-1$? (Answer: the line reflects about the $x$-axis — if the bias is zero.)
- What happens when $b$ goes from $0$ to $+2$? (Answer: the whole line slides up by 2.)

Now verify.

In [ ]:
# Slider + text-box pair, linked in both directions.
def linked_pair(label, value, vmin, vmax, step=0.05):
    slider = FloatSlider(value=value, min=vmin, max=vmax, step=step,
                         description=label, continuous_update=True,
                         readout=False)
    textbox = FloatText(value=value, description=" ", step=step,
                        layout={"width": "110px"})
    jslink((slider, "value"), (textbox, "value"))
    return slider, textbox

w_sl, w_tx = linked_pair("weight w", value=1.0, vmin=-3.0, vmax=3.0)
b_sl, b_tx = linked_pair("bias  b", value=0.0, vmin=-4.0, vmax=4.0, step=0.1)

fig, ax = plt.subplots()
line_plot, = ax.plot(x, w_sl.value * x + b_sl.value, color="#2E86C1", lw=2.2)
ax.axhline(0, color="#7F8C8D", lw=0.6); ax.axvline(0, color="#7F8C8D", lw=0.6)
ax.set_xlim(-4, 4); ax.set_ylim(-6, 6)
ax.set_xlabel("x"); ax.set_ylabel("y = w·x + b")
title = ax.set_title(f"y = {w_sl.value:+.2f}·x {b_sl.value:+.2f}")

def redraw(*_):
    line_plot.set_ydata(w_sl.value * x + b_sl.value)
    title.set_text(f"y = {w_sl.value:+.2f}·x {b_sl.value:+.2f}")
    fig.canvas.draw_idle()

w_sl.observe(redraw, names="value")
b_sl.observe(redraw, names="value")

display(VBox([HBox([w_sl, w_tx]), HBox([b_sl, b_tx])]))

## 4 · Vectors and the dot product

The dot product is *the* operation. Three equivalent ways to compute it:

In [ ]:
a = np.array([3.0, 1.0])
v = np.array([2.0, 2.5])

explicit = a[0]*v[0] + a[1]*v[1]
numpy_dot = np.dot(a, v)
at_operator = a @ v

print(f"explicit sum : {explicit}")
print(f"np.dot(a, v) : {numpy_dot}")
print(f"a @ v        : {at_operator}")
print(f"all equal    : {explicit == numpy_dot == at_operator}")

In [ ]:
# Geometric form:  a · b  =  |a| |b| cos(theta)
norm_a = np.linalg.norm(a)
norm_v = np.linalg.norm(v)
cos_theta = np.dot(a, v) / (norm_a * norm_v)
theta_deg = np.degrees(np.arccos(cos_theta))
print(f"|a| = {norm_a:.3f},   |v| = {norm_v:.3f}")
print(f"cos(theta) = {cos_theta:.3f}  →  theta = {theta_deg:.1f}°")
print(f"|a|·|v|·cos(theta) = {norm_a * norm_v * cos_theta:.3f}  (matches the dot product above)")

**Exercise — perpendicular vectors.** Two vectors are perpendicular when their dot product is zero. Find a vector perpendicular to $\mathbf{a} = [3,\,1]$ without doing any algebra on paper — try candidates in code until `np.dot(a, c)` returns 0.

In [ ]:
# One neat trick: (x, y)  →  (-y, x) is always perpendicular to the original.
c = np.array([-a[1], a[0]])
print(f"a = {a}, c = {c}, a·c = {np.dot(a, c)}")

## 5 · Running example — fit a line to the first 0.2 s of a free throw

Simulate a player's release. Real mechanics will be $h(t) = v_{0y}\,t - \tfrac{1}{2}g\,t^2$ (Ch.2 owns the $-\tfrac{1}{2}gt^2$), but for $t < 0.2$ s the ball rises almost linearly. We add a little measurement noise to mimic a real motion-capture rig.

In [ ]:
v0y_true = 7.2   # true vertical release velocity (m/s)
h0_true = 0.0    # release height taken as origin for this chapter
g = 9.81

t = np.linspace(0, 0.20, 9)
h_ideal = v0y_true * t - 0.5 * g * t ** 2
h_obs   = h_ideal + np.random.normal(0, 0.015, size=t.shape)

fig2, ax2 = plt.subplots()
ax2.scatter(t, h_obs, s=55, color="#2C3E50", zorder=5, label="recorded samples")
ax2.set_xlabel("time  t  (s)"); ax2.set_ylabel("height  h  (m)")
ax2.set_title("Free throw — first 0.2 s after release")
ax2.legend()
plt.show()

### 5.1 · Fit by eye

Move the sliders until the blue line is the best fit you can make. The text boxes accept typed values. The grey number is the total squared miss — smaller is better, but don't look at it *first*: try to eyeball the fit, then check.

In [ ]:
w2_sl, w2_tx = linked_pair("weight  w", value=5.5, vmin=0.0, vmax=12.0, step=0.05)
b2_sl, b2_tx = linked_pair("bias    b", value=0.1, vmin=-0.5, vmax=0.5, step=0.005)
loss_out = Output()

fig3, ax3 = plt.subplots()
ax3.scatter(t, h_obs, s=55, color="#2C3E50", zorder=5, label="recorded")
fit_line, = ax3.plot(t, w2_sl.value * t + b2_sl.value,
                     color="#2E86C1", lw=2.2, label="your fit")
ax3.set_xlabel("time  t  (s)"); ax3.set_ylabel("height  h  (m)")
ax3.set_ylim(-0.1, 1.5); ax3.legend(loc="upper left")

def update_fit(*_):
    pred = w2_sl.value * t + b2_sl.value
    fit_line.set_ydata(pred)
    sse = float(np.sum((pred - h_obs) ** 2))
    with loss_out:
        loss_out.clear_output(wait=True)
        print(f"total squared miss (lower = better):  {sse:.5f}")
    fig3.canvas.draw_idle()

w2_sl.observe(update_fit, names="value")
b2_sl.observe(update_fit, names="value")
update_fit()

display(VBox([HBox([w2_sl, w2_tx]), HBox([b2_sl, b2_tx]), loss_out]))

### 5.2 · Check — what did we just do?

We picked $w$ and $b$ to minimise the *sum of squared misses* between the line $h(t) = w\,t + b$ and the recorded samples. That is called **least-squares fitting**. Computers can pick the exact optimum for us — we'll see this in Pre-Req Ch.5 (as linear algebra) and in ML Ch.1 (as an optimiser). For now, how close did you come by eye?

In [ ]:
your_w, your_b = w2_sl.value, b2_sl.value

# Closed-form least-squares fit of y = w*t + b (no gradient descent, just algebra).
t_mat = np.c_[t, np.ones_like(t)]                  # (N, 2) design matrix
w_opt, b_opt = np.linalg.lstsq(t_mat, h_obs, rcond=None)[0]

print(f"your fit:      w = {your_w:.3f},  b = {your_b:+.3f}")
print(f"least-squares: w = {w_opt:.3f},  b = {b_opt:+.3f}")
print(f"true physics:  v0y = {v0y_true:.3f},  h0 = {h0_true:+.3f}")

## 6 · From one feature to many

Real problems have many features. Predicting the *landing x-position* of a free throw might depend on release speed, release angle, release height, and wind speed. One weight per feature, one bias overall:

$$\hat{y} \;=\; \mathbf{w}\cdot\mathbf{x} + b \;=\; w_1 x_1 + w_2 x_2 + \cdots + w_d x_d + b$$

In [ ]:
# Shot features and a plausible (hand-picked) set of weights — just for the demo.
features     = ["release speed (m/s)", "release angle (deg)", "release height (m)", "wind speed (m/s)"]
x_shot       = np.array([ 7.5,    52.0,   2.10,    1.2])
w_shot       = np.array([ 0.62,    0.09,  0.45,   -0.30])
bias_shot    = -2.10

y_hat = w_shot @ x_shot + bias_shot
print("per-feature contributions:")
for name, xi, wi in zip(features, x_shot, w_shot):
    print(f"  {name:<22}  x = {xi:6.2f}  w = {wi:+.3f}  →  w·x = {wi*xi:+.3f}")
print(f"  bias                                                     = {bias_shot:+.3f}")
print(f"  ----")
print(f"  prediction  ŷ = Σ w·x + b                                 = {y_hat:+.3f} m (landing offset)")

## 7 · Where this reappears

- **Pre-Req Ch.2.** Same equation, but the input is $[x, x^2, x^3, \ldots]$ so we can fit curves with the same linear machinery.
- **Pre-Req Ch.5.** We stack $N$ samples into a matrix $\mathbf{X}$ and solve $\mathbf{X}\mathbf{w} \approx \mathbf{y}$ all at once.
- **ML Ch.1 Linear Regression.** The optimiser that picks $(w, b)$ for you, plus proper error metrics.
- **ML Ch.4 Neural Networks.** A neuron is one line of this notebook, plus a non-linearity.

**Exercise to finish.** Re-run §5.1 but pick a `w` that is obviously too *large* (say 12) and see what the squared miss does. Then make it obviously too small (say 2). The loss should jump sharply in both directions — that asymmetry around the optimum is what gradient descent exploits.